# Berechnung der Differenzen

In [1]:
# Bibliotheken und Funktionen laden
from helper import *

**CSV einlesen & vorbereiten**

In [ ]:
csv_datei = raw_to_csv("../data/Jun/220858_06062025.raw")

# Ziel-Spaltenliste
target_columns = ['T  (s)', 'CL1cBAK2', 'CL2polyAC', 'CL3cBAK2',
                  'CL4polyAC', 'CL5cBAK2', 'CL6polyAC', 'CL7cBAK2',
                  'CL8polyAC', 'Inj.', 'Val.']

# Spalten-Auswahl
spalten_indices = [0, 4, 5, 6, 7, 8, 9, 10, 11, 20, 21]

# CSV einlesen
df = pd.read_csv(csv_datei)

# Nur relevante Spalten auswählen
df = df.iloc[:, spalten_indices].copy()

# Spaltennamen korrigieren
df.columns = [col.replace('CL5McBAK2', 'CL5cBAK2')
                .replace('Cl6polyAC', 'CL6polyAC')  # kleingeschriebenes l korrigieren
              for col in df.columns]

# ❌ "Not used"-Spalten ignorieren
df = df.loc[:, ~df.columns.str.contains("Not used", case=False)]

# Durchgang berechnen anhand der Zeit
min_time = df['T  (s)'].min()
max_time = df['T  (s)'].max()
bins = np.linspace(min_time, max_time, 8)
df['Durchgang'] = pd.cut(
    df['T  (s)'], bins=bins, labels=np.arange(1, 8), include_lowest=True
).astype(int)

# Sensor-Spalten identifizieren
non_sensor_cols = ['T  (s)', 'Inj.', 'Val.', 'Durchgang']
sensor_cols = [col for col in df.columns if col not in non_sensor_cols]

# Sensorwerte bei Val. == 3 auf 0 setzen
df.loc[df['Val.'] == 3, sensor_cols] = 0

starts = df.index[(df['Val.'] == 2) & (df['Val.'].shift(1) == 1)]

for start in starts:
    # Buffer-Bereich rückwärts
    buf_end = start - 1
    buf_start = buf_end
    while buf_start - 1 >= 0 and df.loc[buf_start - 1, 'Val.'] == 1:
        buf_start -= 1

    # Sample-Bereich vorwärts
    end = start
    while end + 1 < len(df) and df.loc[end + 1, 'Val.'] == 2:
        end += 1

    # Offset der ersten Sample-Zeile
    offsets = df.loc[start, sensor_cols]

    # Subtraktion des Offsets
    df.loc[buf_start:end, sensor_cols] = df.loc[buf_start:end, sensor_cols].subtract(offsets, axis=1)

# Kontroll-Ausgabe
reference_columns = target_columns + ['Durchgang']
print("ℹ️ Vorhandene Spalten nach Import:", list(df.columns))

ℹ️ Vorhandene Spalten nach Import: ['T  (s)', 'CL1cBAK2', 'CL2polyAC', 'CL3cBAK2', 'CL4polyAC', 'CL5cBAK2', 'CL6polyAC', 'CL7cBAK2', 'CL8polyAC', 'Inj.', 'Val.', 'Durchgang']


**Differenzen-Plot & Speicherung**

In [ ]:
def plot_sensor_differences(df, filepath=None):
    # Zeit in Minuten
    x = df['T  (s)'] / 60

    # Gewünschte Sensor-Gruppen
    specific_all    = ['CL1cBAK2', 'CL3cBAK2', 'CL5cBAK2', 'CL7cBAK2']
    nonspecific_all = ['CL2polyAC', 'CL4polyAC', 'CL6polyAC', 'CL8polyAC']

    # Nur Sensoren nehmen, die auch wirklich vorhanden sind
    specific    = [s for s in specific_all if s in df.columns]
    nonspecific = [n for n in nonspecific_all if n in df.columns]

    print("✔️ Verwendete spezifische Sensoren:", specific)
    print("✔️ Verwendete unspezifische Sensoren:", nonspecific)

    # Neues DataFrame für die Differenzen
    df_diff = pd.DataFrame()
    df_diff['T_min'] = x
    df_diff['Durchgang'] = df['Durchgang']
    df_diff['Inj.'] = df['Inj.']
    df_diff['Val.'] = df['Val.']

    # 16 positive Differenzen (falls möglich)
    positive_cols = []
    for spec in specific:
        for nonspec in nonspecific:
            if spec in df.columns and nonspec in df.columns:
                colname = f"{spec}_minus_{nonspec}"
                df_diff[colname] = df[spec] - df[nonspec]
                positive_cols.append(colname)

    # 16 negative Differenzen (positive * -1)
    for col in positive_cols:
        df_diff[col + "_neg"] = df_diff[col] * -1

    # ➕ Zusatz-Spalten (nur wenn beide Gruppen existieren!)
    if specific and nonspecific:
        mean_specific = df[specific].mean(axis=1)
        mean_nonspecific = df[nonspecific].mean(axis=1)

        df_diff['mean_diff'] = mean_specific - mean_nonspecific
        df_diff['mean_diff_std'] = (mean_specific - mean_nonspecific).rolling(window=5, min_periods=1).std()

        df_diff['specific_avg_diff'] = df[specific].sub(mean_specific, axis=0).abs().mean(axis=1)
        df_diff['specific_avg_diff_std'] = df[specific].sub(mean_specific, axis=0).abs().std(axis=1)

        df_diff['nonspecific_avg_diff'] = df[nonspecific].sub(mean_nonspecific, axis=0).abs().mean(axis=1)
        df_diff['nonspecific_avg_diff_std'] = df[nonspecific].sub(mean_nonspecific, axis=0).abs().std(axis=1)
    else:
        print("⚠️ Kennzahlen-Berechnung übersprungen (zu wenige Sensoren vorhanden).")


    fig = go.Figure()
    for col in positive_cols:
        fig.add_trace(go.Scatter(
            x=df_diff['T_min'],
            y=df_diff[col],
            mode='lines',
            name=col
        ))

    # Injection Change (rote gestrichelte Linien)
    inj = df_diff['Inj.']
    change_times = [df_diff['T_min'].iloc[i] for i in range(1, len(df_diff)) if inj.iloc[i] != inj.iloc[i-1]]
    ymin, ymax = df_diff[positive_cols].min().min(), df_diff[positive_cols].max().max()
    vline_x, vline_y = [], []
    for ct in change_times:
        vline_x += [ct, ct, None]
        vline_y += [ymin, ymax, None]
    fig.add_trace(go.Scatter(
        x=vline_x, y=vline_y,
        mode='lines',
        line=dict(color='red', width=2, dash='dash'),
        name='Injection Change'
    ))

    # Val-Bereiche (farbige Hintergründe)
    val_colors = {1:'rgba(255,0,0,0.1)', 2:'rgba(0,255,0,0.1)', 3:'rgba(0,0,255,0.1)'}
    current_val = df_diff['Val.'].iloc[0]; start=df_diff['T_min'].iloc[0]
    for i in range(1, len(df_diff)):
        if df_diff['Val.'].iloc[i] != current_val:
            end = df_diff['T_min'].iloc[i]
            fig.add_vrect(x0=start, x1=end,
                          fillcolor=val_colors[current_val],
                          opacity=0.5, layer='below', line_width=0)
            current_val = df_diff['Val.'].iloc[i]
            start = df_diff['T_min'].iloc[i]
    fig.add_vrect(x0=start, x1=df_diff['T_min'].iloc[-1],
                  fillcolor=val_colors[current_val],
                  opacity=0.5, layer='below', line_width=0)

    # Durchgänge (graue Hintergründe mit Beschriftung)
    dg = df_diff.groupby('Durchgang')['T_min'].agg(['min','max']).reset_index()
    shades = ['rgba(200,200,200,0.1)','rgba(150,150,150,0.1)']
    for idx,row in dg.iterrows():
        fig.add_vrect(
            x0=row['min'], x1=row['max'],
            fillcolor=shades[idx%2],
            opacity=0.3, layer='below', line_width=0,
            annotation_text=f"Durchgang {int(row['Durchgang'])}",
            annotation_position="top left"
        )

    # Layout
    fig.update_layout(
        title='Differenzen von spezifisch und unspezifisch über Zeit',
        xaxis_title='Zeit (Minuten)',
        yaxis_title='Differenz (Nanometer)',
        legend_title='Sensor-Paare',
        template='plotly_white'
    )

    fig.show()

    # 👉 Differenzen-Datei speichern
    if filepath is not None:
        base, ext = os.path.splitext(filepath)
        outpath = f"{base}_diff{ext}"
        df_diff.to_csv(outpath, index=False)
        print(f"💾 Differenzen gespeichert in: {outpath}")

# Aufruf
plot_sensor_differences(df, filepath=csv_datei)

✔️ Verwendete spezifische Sensoren: ['CL1cBAK2', 'CL3cBAK2', 'CL5cBAK2', 'CL7cBAK2']
✔️ Verwendete unspezifische Sensoren: ['CL2polyAC', 'CL4polyAC', 'CL6polyAC', 'CL8polyAC']


💾 Differenzen gespeichert in: ../data/Jun/220858_06062025_diff.csv
